# Importando bibliotecas necessárias

In [21]:
from urllib.request import urlopen
from bs4 import BeautifulSoup
import random
import mysql.connector
import re
import requests

In [22]:
dados_conexao = {
    "user":"root",
    "password":"@bmc009@",
    "host":"127.0.0.1",
    "database":"scraping",
    "charset":"utf8mb4"
    }

In [23]:
conexao = mysql.connector.connect(**dados_conexao)
cursor = conexao.cursor()

In [24]:
def gravar (titulo, url, conteudo):
    cursor.execute("insert into paginas (titulo, url, conteudo)"
                   "values (%s, %s, %s)", (titulo, url, conteudo))
    conexao.commit()

In [25]:
def getLinks (urlArtigo):
    url = 'https://pt.wikipedia.org' + urlArtigo

    headers = {'User-Agent':'Mozilla/5.0'}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    # html = urlopen(url)

    bs = BeautifulSoup(response.text, 'html.parser')
    titulo = bs.find('h1').get_text()
    conteudo = bs.find('div', {'id':'mw-content-text'}).find('p').get_text()
    gravar(titulo, url, conteudo)
    return bs.find('div', {'id':'bodyContent'}).findAll('a', href=re.compile('^(/wiki/)((?!:).)*$'))

In [26]:
links = getLinks('/wiki/Copa_do_Mundo_FIFA_de_2026')

try:
    contador = 1
    while len(links) > 0 and contador <= 10:
        novoArtigo = links[random.randint(0, len(links) - 1)].attrs['href']
        # print(str(contador) + " -> " + novoArtigo)
        links = getLinks(novoArtigo)
        contador += 1
finally:
    cursor.close()
    conexao.close()

/tmp/ipykernel_15105/404483861.py:15: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  return bs.find('div', {'id':'bodyContent'}).findAll('a', href=re.compile('^(/wiki/)((?!:).)*$'))
